# Amazon Employee Access: preprocessing




In [5]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_DIR = Path("C:/Case Study Trees/AmazonEmployeeStudy")

RAW_DIR = PROJECT_DIR / "data" / "raw"
SPLIT_DIR = PROJECT_DIR / "data" / "splits"
PREPROCESSOR_DIR = PROJECT_DIR / "artifacts" / "preprocessors"

RANDOM_STATE = 42

train = pd.read_csv(RAW_DIR / "train.csv")
competition_test = pd.read_csv(RAW_DIR / "test.csv")

print("Labelled training file:", train.shape)
print("Competition test file:", competition_test.shape)

print("\nTraining columns:")
print(train.columns.tolist())

print("\nTarget counts:")
print(train["ACTION"].value_counts())

display(competition_test.head())
display(train.head())

Labelled training file: (32769, 10)
Competition test file: (58921, 10)

Training columns:
['ACTION', 'RESOURCE', 'MGR_ID', 'ROLE_ROLLUP_1', 'ROLE_ROLLUP_2', 'ROLE_DEPTNAME', 'ROLE_TITLE', 'ROLE_FAMILY_DESC', 'ROLE_FAMILY', 'ROLE_CODE']

Target counts:
ACTION
1    30872
0     1897
Name: count, dtype: int64


,id,RESOURCE,MGR_ID,ROLE_ROLLUP_1,ROLE_ROLLUP_2,ROLE_DEPTNAME,ROLE_TITLE,ROLE_FAMILY_DESC,ROLE_FAMILY,ROLE_CODE
0,1,78766,72734,118079,118080,117878,117879,118177,19721,117880
1,2,40644,4378,117961,118327,118507,118863,122008,118398,118865
2,3,75443,2395,117961,118300,119488,118172,301534,249618,118175
3,4,43219,19986,117961,118225,118403,120773,136187,118960,120774
4,5,42093,50015,117961,118343,119598,118422,300136,118424,118425


,ACTION,RESOURCE,MGR_ID,ROLE_ROLLUP_1,ROLE_ROLLUP_2,ROLE_DEPTNAME,ROLE_TITLE,ROLE_FAMILY_DESC,ROLE_FAMILY,ROLE_CODE
0,1,39353,85475,117961,118300,123472,117905,117906,290919,117908
1,1,17183,1540,117961,118343,123125,118536,118536,308574,118539
2,1,36724,14457,118219,118220,117884,117879,267952,19721,117880
3,1,36135,5396,117961,118343,119993,118321,240983,290919,118322
4,1,42680,5905,117929,117930,119569,119323,123932,19793,119325


In [7]:
from sklearn.model_selection import train_test_split

data = train.copy()
data["ROW_ID"] = np.arange(len(data))

train_data, remaining_data = train_test_split(
    data,
    test_size=0.30,
    stratify=data["ACTION"],
    random_state=RANDOM_STATE
)

valid_data, test_data = train_test_split(
    remaining_data,
    test_size=0.50,
    stratify=remaining_data["ACTION"],
    random_state=RANDOM_STATE
)

print("Training rows:", len(train_data))
print("Validation rows:", len(valid_data))
print("Test rows:", len(test_data))

print("\nTarget proportions:")
display(
    pd.DataFrame({
        "Train": train_data["ACTION"].value_counts(normalize=True),
        "Validation": valid_data["ACTION"].value_counts(normalize=True),
        "Test": test_data["ACTION"].value_counts(normalize=True)
    }).round(4)
)

Training rows: 22938
Validation rows: 4915
Test rows: 4916

Target proportions:


,Train,Validation,Test
ACTION,,,
1,0.9421,0.9422,0.942
0,0.0579,0.0578,0.058


In [8]:

train_data.to_csv(SPLIT_DIR / "train.csv", index=False)
valid_data.to_csv(SPLIT_DIR / "valid.csv", index=False)
test_data.to_csv(SPLIT_DIR / "test.csv", index=False)


X_train = train_data.drop(columns=["ACTION", "ROW_ID"]).copy()
X_valid = valid_data.drop(columns=["ACTION", "ROW_ID"]).copy()
X_test = test_data.drop(columns=["ACTION", "ROW_ID"]).copy()

y_train = train_data["ACTION"].copy()
y_valid = valid_data["ACTION"].copy()
y_test = test_data["ACTION"].copy()

categorical_columns = X_train.columns.tolist()

for data in [X_train, X_valid, X_test]:
    for column in categorical_columns:
        data[column] = data[column].map(
            lambda value: str(value) if pd.notna(value) else np.nan
        )

print("Categorical predictors:", len(categorical_columns))
print(categorical_columns)

print("\nTraining missing values:")
print(X_train.isna().sum())

display(X_train.head())

Categorical predictors: 9
['RESOURCE', 'MGR_ID', 'ROLE_ROLLUP_1', 'ROLE_ROLLUP_2', 'ROLE_DEPTNAME', 'ROLE_TITLE', 'ROLE_FAMILY_DESC', 'ROLE_FAMILY', 'ROLE_CODE']

Training missing values:
RESOURCE            0
MGR_ID              0
ROLE_ROLLUP_1       0
ROLE_ROLLUP_2       0
ROLE_DEPTNAME       0
ROLE_TITLE          0
ROLE_FAMILY_DESC    0
ROLE_FAMILY         0
ROLE_CODE           0
dtype: int64


,RESOURCE,MGR_ID,ROLE_ROLLUP_1,ROLE_ROLLUP_2,ROLE_DEPTNAME,ROLE_TITLE,ROLE_FAMILY_DESC,ROLE_FAMILY,ROLE_CODE
13347,73110,4945,117961,118300,118360,118129,122064,118131,118132
2312,13878,13037,118006,118007,118008,117879,117897,19721,117880
8191,40855,7560,117961,118300,119195,117905,117906,290919,117908
3940,41289,71401,91261,118026,118202,118784,124402,290919,118786
20608,17183,4828,117961,118343,118395,118890,125128,118398,118892


In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
import joblib

onehot_preprocessor = ColumnTransformer([
    (
        "categorical",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True,
            dtype=np.float32
        ),
        categorical_columns
    )
])

onehot_preprocessor.fit(X_train)

joblib.dump(
    onehot_preprocessor,
    PREPROCESSOR_DIR / "rf_onehot_preprocessor.joblib"
)

print("Original features:", X_train.shape[1])
print(
    "One-hot features:",
    len(onehot_preprocessor.get_feature_names_out())
)

Original features: 9
One-hot features: 13694
